<!-- TARGET: public course-materials repo, sessions/m05/lab.ipynb. Copy to work/m05/ before editing. -->
# Module 5 · Lab: the order mart

**Python for Data Science · ESADE MIBA · Wed 11 Nov 2026 · homework, before M06**

Copy this notebook to `work/m05/lab.ipynb` and work on the copy. About an hour.

You build `data/mart/orders_enriched.parquet`: **one row per order**, with the customer, the
items, the payments and the review. Every chart in M06–M07 and the case in M08 start from it.

Fill in each cell with `...`, then run the **check** cell below it. Silence means correct; an
`AssertionError` tells you what to look at.

Two revenue numbers, used the same way in every notebook from now on:

- **Paid revenue** = `payment_value`, summed per order. What customers paid.
- **Item revenue** = `price + freight_value`, summed over item lines. What each item sold for.
  The only one that can be split by seller or category.

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("../../data/raw")
CLEAN = Path("../../data/clean")
MART = Path("../../data/mart")
CLEAN.mkdir(parents=True, exist_ok=True)
MART.mkdir(parents=True, exist_ok=True)

DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
if not (CLEAN / "orders.parquet").exists():
    pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=DATE_COLUMNS).to_parquet(
        CLEAN / "orders.parquet", index=False
    )
orders = pd.read_parquet(CLEAN / "orders.parquet")

The cell above writes the clean orders if you missed the demo. Nothing to do there.

## Part 1 · The clean layer (15 min)

**1a.** Write `items`, `payments` and `customers` to `data/clean/` as `items.parquet`,
`payments.parquet` and `customers.parquet`. In `customers`, keep `customer_zip_code_prefix`
as text (the M04 trap: the leading zero).

In [ ]:
items = ...
payments = ...
customers = ...

In [ ]:
assert len(pd.read_parquet(CLEAN / "items.parquet")) == 112650
assert len(pd.read_parquet(CLEAN / "payments.parquet")) == 103886
assert pd.read_parquet(CLEAN / "customers.parquet")["customer_zip_code_prefix"].str.len().eq(5).all()

**1b.** Write `reviews` to `data/clean/reviews.parquet` with `review_creation_date` and
`review_answer_timestamp` parsed as dates.

In [ ]:
reviews = ...

In [ ]:
check = pd.read_parquet(CLEAN / "reviews.parquet")
assert len(check) == 99224
assert str(check["review_answer_timestamp"].dtype).startswith("datetime64")

## Part 2 · Grain (15 min)

**2a.** Write one `assert` per table proving its grain: `orders` (one order), `customers` (one
customer per order), `items` (one item line), `payments` (one payment).

In [ ]:
...

**2b.** Reviews are **not** one per order. Keep only the **latest** review of each order (the
one with the most recent `review_answer_timestamp`) in `latest_reviews`. Then prove the new
grain with an `assert`.

In [ ]:
latest_reviews = ...

In [ ]:
assert latest_reviews["order_id"].is_unique
assert len(latest_reviews) == reviews["order_id"].nunique() == 98673

**2c.** In one sentence: why `drop_duplicates(subset="order_id")` on its own, without sorting
first, is not good enough.

*Your answer:* ...

## Part 3 · Change the grain (10 min)

**3a.** Build `items_by_order`, one row per order, with named aggregations: `n_items` (number
of item lines) and `item_revenue` (sum of `price + freight_value`).

In [ ]:
items_by_order = ...

In [ ]:
assert items_by_order["order_id"].is_unique
assert items_by_order["n_items"].sum() == len(items)
assert round(items_by_order["item_revenue"].sum(), 2) == 15843553.24

**3b.** Build `paid_by_order`, one row per order, with `paid_revenue`.

In [ ]:
paid_by_order = ...

In [ ]:
assert paid_by_order["order_id"].is_unique
assert round(paid_by_order["paid_revenue"].sum(), 2) == 16008872.12

## Part 4 · Assemble the mart (20 min)

**4a.** Start from `orders` and **left**-join, each with `validate="one_to_one"`:

1. `customers[["customer_id", "customer_unique_id", "customer_state"]]` on `customer_id`
2. `items_by_order` on `order_id`, with `indicator="items_merge"`
3. `paid_by_order` on `order_id`
4. `latest_reviews[["order_id", "review_score"]]` on `order_id`

Call the result `mart`.

In [ ]:
mart = ...

In [ ]:
assert len(mart) == len(orders) == 99441
assert mart["order_id"].is_unique
assert (mart["items_merge"] == "left_only").sum() == 775
assert round(mart["paid_revenue"].sum(), 2) == 16008872.12

**4b.** Add the derived columns and keep only the mart's columns:

- `has_items`: `True` if the order has item lines (from `items_merge`)
- `n_items` and `item_revenue`: `0` for orders without items
- `purchase_month`: `"YYYY-MM"` text, from `order_purchase_timestamp`
- `delivery_days`: as in M04, `(delivered − purchased).dt.days`
- `is_late`: delivered on a **later day** than estimated (compare days, as in M04); missing
  where there is no delivery date

Columns, in this order: `order_id`, `customer_unique_id`, `customer_state`, `order_status`,
`order_purchase_timestamp`, `purchase_month`, `delivery_days`, `is_late`, `n_items`,
`item_revenue`, `paid_revenue`, `has_items`, `review_score`.

In [ ]:
MART_COLUMNS = [
    "order_id", "customer_unique_id", "customer_state", "order_status", "order_purchase_timestamp",
    "purchase_month", "delivery_days", "is_late", "n_items", "item_revenue", "paid_revenue",
    "has_items", "review_score",
]
mart = ...

In [ ]:
assert list(mart.columns) == MART_COLUMNS
assert mart["has_items"].sum() == 98666
assert mart["n_items"].sum() == 112650
assert mart["is_late"].isna().sum() == orders["order_delivered_customer_date"].isna().sum()
delivered = mart[mart["order_status"] == "delivered"]
assert round(delivered["is_late"].mean(), 3) == 0.068

**4c.** Write the mart to `data/mart/orders_enriched.parquet`, read it back, and check that
nothing changed.

In [ ]:
...

In [ ]:
from_disk = pd.read_parquet(MART / "orders_enriched.parquet")
assert from_disk.shape == (99441, 13)
assert from_disk["order_id"].is_unique
assert str(from_disk["order_purchase_timestamp"].dtype).startswith("datetime64")

## Stretch · Paid ≠ item revenue

For orders that have both items and payments, how many differ by more than 1 BRL between
`paid_revenue` and `item_revenue`? Store it in `n_mismatch`. Look at a few: which payment
types do they use? (Hint: join `payments` back on.)

In [ ]:
n_mismatch = ...

In [ ]:
assert n_mismatch == 249

## Done?

Next week the mart becomes charts. If this notebook doesn't reach the end, the solution is
released at the start of M06: running it writes the same mart.